# Memory Decay — Why AI Agents Forget User Preferences

Based on:
- [MemoryOS of AI Agent](https://arxiv.org/abs/2506.06326) — Kang et al., 2025
- [Cognitive Memory in Large Language Models](https://arxiv.org/abs/2504.02441) — Shan et al., 2025

## The Problem

When an AI agent interacts with a user across multiple turns, it has no built-in mechanism to **learn from past interactions**. Each turn is processed independently — the agent cannot build a user profile, remember preferences, or personalize future responses.

Real-world impact:
- A travel assistant recommends budget hostels after the user booked a luxury ryokan
- A shopping agent suggests items in the wrong size every time
- A support agent asks for the same account details on every interaction

## The Solution: agent.state + SessionManager

Strands Agents provides two mechanisms:
1. **`agent.state`** — key-value store on the agent instance, persists across turns within a session
2. **`FileSessionManager`** / **`S3SessionManager`** — persists state across sessions (agent restarts)

Tools use `@tool(context=True)` to access `agent.state` via `ToolContext`, learning preferences from user actions and using them to personalize future responses.

## What We Test

| Test | Approach | Remembers preferences | Cross-session |
|------|----------|----------------------|---------------|
| 1 — Stateless | No memory tools | No | No |
| 2 — Stateful | `agent.state` tools | Yes | No |
| 3 — Persistent | `agent.state` + `FileSessionManager` | Yes | Yes |

## Configure API Key

Set your OpenAI API key. Get one at https://platform.openai.com/api-keys

> You can swap to any supported model provider. Change the model in the setup cell below.

In [ ]:
import os
# os.environ['OPENAI_API_KEY'] = 'your-key-here'  # Uncomment and set your key
assert os.getenv('OPENAI_API_KEY'), (
    'OPENAI_API_KEY not set. '
    'Get yours at https://platform.openai.com/api-keys and set it above or in a .env file.'
)

## Setup

In [ ]:
# This demo uses Strands Agents SDK. The memory patterns apply to any agent framework with state management.
import json, time, os

os.environ['OTEL_SDK_DISABLED'] = 'true'

from dotenv import load_dotenv
from strands import Agent, tool, ToolContext
# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel
from strands.agent.conversation_manager import SlidingWindowConversationManager
from strands.session import FileSessionManager
from tools import (
    search_hotels_stateless, book_hotel_stateless,
    search_hotels, book_hotel, get_user_profile,
)

load_dotenv()

MODEL = OpenAIModel(model_id='gpt-4o-mini')

SYSTEM_PROMPT = (
    'You are a travel assistant. Help users find and book hotels. '
    'When the user books a hotel, remember their preferences (style, stars, amenities) '
    'and use them to rank results in future searches. '
    'Always be concise — answer in 2-3 sentences maximum.'
)

# Same conversation for all tests
TURN_1 = 'Search for hotels in Tokyo under $350/night'
TURN_2 = 'Book the Zen Garden Ryokan for 3 nights'
TURN_3 = 'Now search for hotels in Zurich — what do you recommend based on what you know about me?'

print('Setup complete!')

---
## Test 1 — Stateless Agent (No Memory)

The stateless agent uses tools that have **no access to `agent.state`** (a key-value store for persisting data across conversation turns). Each tool call returns results without learning anything about the user.

**Conversation:**
1. Search Tokyo hotels
2. Book a traditional 4-star ryokan with onsen and garden
3. Search Zurich hotels — does the agent personalize results?

**Expected:** Turn 3 returns generic results. The agent has no knowledge of the user's preference for traditional, 4-star properties with cultural amenities.

In [ ]:
agent_stateless = Agent(
    model=MODEL,
    system_prompt=SYSTEM_PROMPT,
    tools=[search_hotels_stateless, book_hotel_stateless],
)

print('Turn 1:', TURN_1)
agent_stateless(TURN_1)

print('\nTurn 2:', TURN_2)
agent_stateless(TURN_2)

print('\nTurn 3:', TURN_3)
agent_stateless(TURN_3)

# Check state
prefs = agent_stateless.state.get('user_preferences')
print(f'\nagent.state["user_preferences"]: {prefs}')
print('Result: No preferences learned. Agent treats every search as a new user.')

---
## Test 2 — Stateful Agent (agent.state Memory)

The stateful agent uses tools with `@tool(context=True)` that access `agent.state` via `ToolContext`. When the user books a hotel, the tool extracts preferences (style, stars, amenities) and stores them.

On the next search, the tool reads preferences and **ranks results accordingly**.

```python
@tool(context=True)
def book_hotel(hotel_name: str, tool_context: ToolContext, nights: int = 1) -> str:
    # ... book the hotel ...
    prefs = tool_context.agent.state.get('user_preferences') or {}
    prefs['preferred_style'] = hotel['style']      # 'traditional'
    prefs['preferred_stars'] = hotel['stars']        # 4
    prefs['preferred_amenities'] = hotel['amenities'] # ['onsen', 'garden', 'restaurant']
    tool_context.agent.state.set('user_preferences', prefs)
```

**Expected:** Turn 3 ranks Zurich hotels by similarity to the Tokyo ryokan — traditional/lodge style, 4+ stars, cultural amenities.

In [ ]:
agent_stateful = Agent(
    model=MODEL,
    system_prompt=SYSTEM_PROMPT,
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=[search_hotels, book_hotel, get_user_profile],
)

print('Turn 1:', TURN_1)
agent_stateful(TURN_1)

print('\nTurn 2:', TURN_2)
agent_stateful(TURN_2)

# Show learned preferences
prefs = agent_stateful.state.get('user_preferences')
print(f'\nLearned preferences after booking:')
print(json.dumps(prefs, indent=2))

print('\nTurn 3:', TURN_3)
agent_stateful(TURN_3)

history = agent_stateful.state.get('booking_history')
print(f'\nBooking history: {len(history)} bookings')
print('Result: Zurich results ranked by similarity to Tokyo ryokan preferences.')

---
## Test 3 — Session Persistence (FileSessionManager)

`agent.state` solves within-session memory, but what about **returning users**? When the agent restarts (new Python process, server reboot), `agent.state` is lost.

`FileSessionManager` persists state to disk. On the next session with the same `session_id`, the agent restores its state — including all learned preferences.

```python
agent = Agent(
    model=MODEL,
    tools=[search_hotels, book_hotel],
    session_manager=FileSessionManager(
        session_id='travel-user-42',
        storage_dir='./sessions'
    ),
)
```

**Expected:** Session B (new agent instance) restores preferences from Session A without re-asking.

In [ ]:
import shutil

session_id = 'travel-user-demo'
storage_dir = os.path.join(os.path.dirname(os.path.abspath('.')), 'sessions')

# --- Session A: First visit ---
print('--- Session A: First visit ---')
agent_a = Agent(
    model=MODEL,
    system_prompt=SYSTEM_PROMPT,
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=[search_hotels, book_hotel, get_user_profile],
    session_manager=FileSessionManager(session_id=session_id, storage_dir=storage_dir),
)

agent_a(TURN_1)
agent_a(TURN_2)

prefs_a = agent_a.state.get('user_preferences')
print(f'\nPreferences after Session A: {json.dumps(prefs_a)}')

# --- Session B: Returning user (new agent instance) ---
print('\n--- Session B: Returning user (new agent, same session_id) ---')
agent_b = Agent(
    model=MODEL,
    system_prompt=SYSTEM_PROMPT,
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=[search_hotels, book_hotel, get_user_profile],
    session_manager=FileSessionManager(session_id=session_id, storage_dir=storage_dir),
)

prefs_b = agent_b.state.get('user_preferences')
print(f'Preferences restored in Session B: {json.dumps(prefs_b)}')
print(f'State survived restart: {prefs_a == prefs_b}')

print('\nTurn 3 (in new session):', TURN_3)
agent_b(TURN_3)

# Cleanup
if os.path.exists(storage_dir):
    shutil.rmtree(storage_dir)
    print('\n(Session files cleaned up)')

---
## Comparison

| Test | Remembers preferences | Cross-session | Personalized results |
|------|----------------------|---------------|---------------------|
| 1 — Stateless | No | No | No |
| 2 — Stateful (`agent.state`) | Yes | No | Yes |
| 3 — Persistent (`FileSessionManager`) | Yes | Yes | Yes |

## Key Takeaways

- Without `agent.state`, the agent has **no mechanism to learn** from user actions
- `agent.state` enables within-session personalization with zero extra infrastructure
- `FileSessionManager` / `S3SessionManager` extends this across sessions — the agent remembers returning users
- Tools must be explicitly designed to **write** preferences and **read** them for personalization

## Next Steps

1. [Demo 02: Core Memory Pattern](../02-core-memory-demo/) — Structured memory with explicit read/write/update tools
2. [Demo 03: Memory Retrieval](../03-memory-retrieval-demo/) — When memory grows large, find the right memories

## References

- [MemoryOS of AI Agent](https://arxiv.org/abs/2506.06326) — Hierarchical memory management (+49% F1)
- [Cognitive Memory in LLMs](https://arxiv.org/abs/2504.02441) — Survey of sensory, short-term, long-term memory
- [MemGPT: Towards LLMs as Operating Systems](https://arxiv.org/abs/2310.08560) — Virtual context management
- [Strands Agent State](https://github.com/strands-agents/sdk-python)
- [Strands Session Management](https://github.com/strands-agents/sdk-python)
- [Code Repository](https://github.com/aws-samples/sample-why-agents-fail)